# 🐘 hadoop04_mapreduce_yarn.ipynb
### Phase 7 — Hadoop · Part 4
**Topics:** MapReduce Concepts · YARN Architecture
**Note:** Concepts only. No Java practicals. Spark does this better.

---
```
Videos to watch before filling this:
MapReduce: 96, 101, 104, 105
YARN:      107, 108, 109, 110
```


## 🗣️ EXPLAIN BLOCK — Fill this LAST

**Q1: What problem does MapReduce solve in one sentence?**

`YOUR ANSWER:`

**Q2: What does Map phase do? What does Reduce phase do?**

`YOUR ANSWER:`

**Q3: What is a key-value pair? Give a real Word Count example.**

`YOUR ANSWER:`

**Q4: What is YARN and why does Hadoop need it?**

`YOUR ANSWER:`

**Q5: Why did Spark replace MapReduce? One sentence.**

`YOUR ANSWER:`


---
---
# SECTION 1 — MapReduce
---


## ⌨️ BUILD 1 — The Core Problem MapReduce Solves

**Q: Before MapReduce, how did companies process 1TB of data?**

`YOUR ANSWER:`

**Q: Fill the analogy:**
```
Single machine approach:
  1 server reads 1TB → very slow, maybe crashes

MapReduce approach:
  100 machines each read 10gb in parallel
  Each machine processes its own piece
  Results combined at the end
  Total time = veryless
```

**Q: What are the 2 phases and what does each do?**
```
Map Phase    → input: one block of data
               output: data in Key value pair

Reduce Phase → input: grouped key-value pairs
               output: aggregate the values


---
## ⌨️ BUILD 2 — Key-Value Pairs

> Everything in MapReduce is key-value. This is the foundation.

**Q: What is a key-value pair? Give 3 real DE examples:**
```
Word Count:    key = "the"     value = 200
Sales total:   key = Television     value = 50
Log analysis:  key = input     value = 20
```

**Q: Input line: `the cat sat on the mat`
What does Mapper output for this line?**
```
(the, 1)
(cat, 1)
(sat, 1)
(on, 1)
(the, 1)
(mat, 1)
```


---
## ⌨️ BUILD 3 — Word Count End to End

**Input (2 HDFS blocks):**
```
Block 1: the cat sat on the mat
Block 2: the cat in the hat
```

**Step 1 — Map Phase:**
```
How many Mappers? _____  (one per block)

Mapper 1 output:          Mapper 2 output:
(the, 1)                  (the, 1)
(cat, 1)                  (cat, 1)
(sat, 1)                (in,  1)
(on, 1)                (the, 1)
(the, 1)                  (hat, 1)
(mat, 1)
```

**Step 2 — Shuffle and Sort:**
```
What shuffle does: group all the same keys
After shuffle:
  (the,  [1, 1, 1, 1])  → goes to Reducer (no idea may be the)
  (cat,  [1, 1])        → goes to Reducer _____
  (sat,  [1])           → goes to Reducer _____
  (hat,  [1])           → goes to Reducer _____
  (in,   [1])           → goes to Reducer _____
  (on,   [1])           → goes to Reducer _____
  (mat,  [1])           → goes to Reducer _____
```

**Step 3 — Reduce Phase:**
```
(the,  [1,1,1,1]) → (the,  4)
(cat,  [1,1])     → (cat,  2)
(sat,  [1])       → (sat,  1)
(hat,  [1])       → (hat,  1)
```

**Where is output stored?** `YOUR ANSWER:`HDFS


---
## ⌨️ BUILD 4 — Combiner (from video 101)

**Q: What is a Combiner? Where does it run?**

`YOUR ANSWER:`combiner send the key value pairby combining keys and send to reducer

**Q: Why does it reduce network traffic?**
```
WITHOUT Combiner — Block 1 sends over network:
(the,1) (cat,1) (sat,1) (on,1) (the,1) (mat,1)
= 6 pairs

WITH Combiner — Block 1 sends over network:
(the,2) (cat,1) (sat,1) (on,1) (mat,1)
= 5 pairs

Network traffic reduced by: combiner
```


---
## ⌨️ BUILD 5 — Input Split (from video 104)

**Q: What is an Input Split?**

`YOUR ANSWER:`

**Q: Is Input Split the same as HDFS block?**

`YOUR ANSWER:`

**Q: File = 500MB, Block = 128MB. How many Mappers?**

`YOUR ANSWER:`


In [0]:
# ── BUILD 6 — Word Count in Python (MapReduce Pattern) ──────────────
# Run this. Understand each phase. This is exactly what Hadoop does.

from collections import defaultdict
from functools import reduce

block1 = 'the cat sat on the mat'
block2 = 'the cat in the hat'

# MAP PHASE
def mapper(block):
    return [(word, 1) for word in block.split()]

map1 = mapper(block1)
map2 = mapper(block2)
print('MAP OUTPUT:')
print('Mapper 1:', map1)
print('Mapper 2:', map2)

# SHUFFLE PHASE
def shuffle(map_outputs):
    grouped = defaultdict(list)
    for pairs in map_outputs:
        for key, val in pairs:
            grouped[key].append(val)
    return dict(grouped)

shuffled = shuffle([map1, map2])
print('\nSHUFFLE OUTPUT:')
for k, v in sorted(shuffled.items()):
    print(f'  {k}: {v}')

# REDUCE PHASE
result = {word: sum(counts) for word, counts in shuffled.items()}
print('\nFINAL WORD COUNT:')
for word, count in sorted(result.items(), key=lambda x: -x[1]):
    print(f'  {word}: {count}')


## 🧠 THINK — MapReduce

**Q: In Python above, everything runs on 1 machine. In Hadoop, what runs on different machines?**

`YOUR ANSWER:`mapper

**Q: If you have 1 billion words across 100 blocks — how many Mappers run? Are they parallel?**

`YOUR ANSWER:`100  mappers,yes it will be parallel

**Q: MapReduce writes results to disk after every step. Why is this slow for ML algorithms?**

`YOUR ANSWER:`no idea


---
## ⌨️ BUILD 7 — MapReduce vs Spark

| Factor | MapReduce | Spark |
|--------|-----------|-------|
| Storage during processing | Disk after every step | |
| Speed | | 10-100x faster |
| Multi-step jobs | New job for each step | |
| Language | Java only | |
| ML support | Poor | |
| Still used for | | |

**Q: Your tutor said MapReduce is mostly replaced. What part still runs in production?**

`YOUR ANSWER:`


---
---
# SECTION 2 — YARN
---
> Videos: 107, 108, 109, 110


## ⌨️ BUILD 8 — What is YARN and Why

**Q: What does YARN stand for?**

`YOUR ANSWER:`yet another resource manager

**Q: Before YARN, Hadoop 1.0 had a problem. What was it?**

`YOUR ANSWER:`

**Q: YARN is called 'the OS of Hadoop cluster'. What does that mean?**

`YOUR ANSWER:`manages resources for hadoop

**Q: Fill the analogy from video 109:**
```
YARN is like resource manager in real life
ResourceManager is like _____________
NodeManager is like _____________
ApplicationMaster is like _____________
```


---
## ⌨️ BUILD 9 — YARN Components (video 108)

| Component | Where it runs | What it does |
|-----------|--------------|-------------|
| ResourceManager | Master node |which dn is free where block is available|
| NodeManager | Each DataNode |no idea|
| ApplicationMaster |datanode|coordinate whole job|
| Container | Data node|stores and process the mapper (guess)|

**Q: What is a Container in YARN?**

`YOUR ANSWER:`stores amount of ram and cpu in data node

**Q: ResourceManager has 2 components. What are they?**
```
1. Scheduler        → does:allocates the resource which d0 will do what
2. ApplicationsManager → does: accepsts application or task
```


---
## ⌨️ BUILD 10 — YARN Process Step by Step (video 110)

**Client submits a MapReduce job. Fill every step:**
```
Step 1: Client submits job to _____________

Step 2: ResourceManager allocates a Container
        and starts _____________ inside it

Step 3: ApplicationMaster registers with _____________

Step 4: ApplicationMaster requests _____________ from ResourceManager
        for running Map and Reduce tasks

Step 5: ResourceManager allocates _____________
        on different NodeManagers

Step 6: ApplicationMaster launches _____________
        inside each allocated Container

Step 7: Each task reports progress to _____________

Step 8: When all tasks complete, ApplicationMaster
        reports to _____________ and releases Containers
```


---
## ⌨️ BUILD 11 — YARN Architecture Diagram

> Draw the full architecture as ASCII. Include all components.

```
YOUR DIAGRAM:

         CLIENT
           |
           ↓
   [task to resource manager]     ← what is this?
   /                       \
  ↓                         ↓
[application manager]    [no idea and again confusing]   ← what are these?
  |               |
[Container]   [Container]  ← what runs inside?
```

**Q: Can YARN run Spark jobs too? Or only MapReduce?**

`YOUR ANSWER:`


---
## 🧠 THINK — YARN

**Q: NodeManager dies. What happens to the tasks running on it?**

`YOUR ANSWER:`no idea may be another node manager takes responsibilyrt

**Q: ResourceManager dies. What happens to the whole cluster?**

`YOUR ANSWER:`

**Q: Why is YARN important for Spark? (This connects to next section.)**

`YOUR ANSWER:`


---
---
# SECTION 3 — Bridge to Spark
---
> Understand this before starting Section 17.


## ⌨️ BUILD 12 — How Spark Improves MapReduce

```
MapReduce job with 5 steps:
  Step 1 → writes to HDFS
  Step 2 → reads from HDFS → writes to HDFS
  Step 3 → reads from HDFS → writes to HDFS
  Step 4 → reads from HDFS → writes to HDFS
  Step 5 → reads from HDFS → writes to HDFS
  = 10 disk reads + 10 disk writes

Spark job with 5 steps:
  Step 1 → keeps in RAM
  Step 2 → keeps in RAM
  Step 3 → keeps in RAM
  Step 4 → keeps in RAM
  Step 5 → writes final result to disk
  = _____ disk read + _____ disk write
```

**Q: This is why Spark is faster. What is this RAM-based processing called in Spark?**

`YOUR ANSWER:`

**Q: Does Spark still use YARN?**

`YOUR ANSWER:`

**Q: Does Spark still use HDFS?**

`YOUR ANSWER:`


---
---
# ✅ COMPLETION CHECKLIST

| Task | Done? |
|------|-------|
| 👁️ Watched: 96. Map Reduce & Cluster | |
| 👁️ Watched: 101. Combiner in MR | |
| 👁️ Watched: 104. Input Split in MR | |
| 👁️ Watched: 105. Map Reduce Outro | |
| 👁️ Watched: 107. YARN Introduction | |
| 👁️ Watched: 108. Components of YARN | |
| 👁️ Watched: 109. YARN Analogy | |
| 👁️ Watched: 110. YARN Process Step by Step | |
| ⌨️ BUILD 1 — MapReduce core concept | |
| ⌨️ BUILD 2 — Key-value pairs | |
| ⌨️ BUILD 3 — Word Count traced | |
| ⌨️ BUILD 4 — Combiner | |
| ⌨️ BUILD 5 — Input Split | |
| ⌨️ BUILD 6 — Python simulation (run it) | |
| ⌨️ BUILD 7 — MapReduce vs Spark table | |
| ⌨️ BUILD 8 — YARN what and why | |
| ⌨️ BUILD 9 — YARN components table | |
| ⌨️ BUILD 10 — YARN process steps | |
| ⌨️ BUILD 11 — YARN diagram | |
| ⌨️ BUILD 12 — Bridge to Spark | |
| 🗣️ All 5 EXPLAIN questions answered | |

---
**Checklist full → send back → get spark01_introduction.ipynb 🚀**

> ⚡ Spark starts from Section 17. That's where the real DE work begins.
